# Recol·lecció d'entitats

Produeix `lab/entitats/entidades_candidatas.json`: les entitats que el dataset ha de
reforçar. Tres fonts:

- **A — errors reals** (`python3 src/evaluate.py`): entitats del ground truth de RNE que
  `large-v3` transcriu malament. Ja porten la seva evidència i entren sense round-trip.
- **B — NER sobre corpus** (`es_core_news_md`): *generador* de candidats. Només entren
  les que `src/verify_entities.py` mesura que fallen de veritat.
- **C — fragmentació del tokenizer**: prior barat per **ordenar** els candidats de B.

El LLM confirma la grafia i tipifica l'entitat, però **no** decideix si val la pena
reforçar-la: això ho fa el round-trip, amb àudio.

**Ordre**: executa el notebook sencer → `dictionary.ipynb` (`ETAPA='treball'`) →
`src/verify_entities.py` → **si `entidades_fuente_a_omitidas.json` no és buit**,
també `src/verify_entities.py --input lab/entitats/entidades_fuente_a_omitidas.json`
(mateix fitxer de sortida: les dues crides s'acumulen) → torna aquí i re-executa
**només l'última cel·la** (llegeix de fitxers, no del kernel) → `dictionary.ipynb`
(`ETAPA='final'`).


In [2]:
import json
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path

ARREL = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/llm.py").exists())
sys.path.insert(0, str(ARREL / "src"))
import llm
import phonetics
from evaluate import (OMITIDA, PAIRS, STOP, load_raw, norm, similitud,
                      diagnosticar_cambio, strip_accents)

ENT = ARREL / "lab/entitats"
# Compartida amb A.3 (escriu) i la fusio final (llegeix la provinenca via `font_input`).
RUTA_A_OMITIDES = ENT / "entidades_fuente_a_omitidas.json"
MODEL_VALIDACION = "gpt-4.1-mini"  # API de pagament, sense benchmark propi
candidatos = defaultdict(lambda: {"fuentes": set(), "freq": 0})


def frases_de(textos):
    return [f.strip() for t in textos for f in re.split(r"(?<=[.!?])\s+", t) if f.strip()]


## Font A — errors reals del model

L'ordre és **reconstruir (A.2) i després filtrar (A.3)**, no al revés. `evaluate.py`
alinea per regió i lliura fragments (`comunitat`, `radio`, `pol`), i les regles de
l'embut —nom comú, longitud mínima, soroll d'alineament— només són vàlides sobre
l'entitat sencera. Filtrant primer es perdien `Comunitat Valenciana` (que Whisper
escriu "comunidad"), `Radio Nacional`, `Televisión Española`, `Pol Pot` i `Vox`.

Dins de l'embut, la separació que importa és entre **variant** i **entitat**: una
variant que no és cap error (`diez` → `10`, `paris` → `parís`) es treu tota sola i la
resta d'evidència es conserva. Amb un `any()` sobre el registre sencer, `alaves` →
`['alavés', 'a la vez']` queia sencera i es perdia la fallada real.


In [3]:
# A.1 — Errors detectats i senyal de "nom comú": si una paraula surt TAMBÉ en minúscula
# al ground truth, és un nom comú capitalitzat per posició (Radio, Día) i no una entitat.
errores = json.loads((ENT / "entitats_fallades/entidades_erroneas.json").read_text("utf-8"))
textos_gt = [load_raw(gt) for _n, _s, gt in PAIRS]
frases_gt = frases_de(textos_gt)

uso_mayus, uso_minus = Counter(), Counter()
for frase in frases_gt:
    for i, palabra in enumerate(frase.split()):
        tok = palabra.strip(',.;:¿?¡!()"\'«»…—-')
        if tok and i > 0 and (n := norm(tok)):        # i==0 sempre va capitalitzada
            (uso_mayus if tok[0].isupper() else uso_minus)[n] += 1

print(f"{len(errores)} entitats amb error / "
      f"{sum(sum(n for _, n in e['mal_transcrita']) for e in errores)} errors totals")


338 entitats amb error / 609 errors totals


In [4]:
# A.2 — Reconstruir entitats multiparaula. Alinear token a token parteix els noms
# compostos i deixa fragments solts ('torre', 'corts'); busquem al GT seqüències de
# paraules capitalitzades seguides per recuperar la forma completa.
MAX_TOKENS = 3               # més de 3 gairebé sempre ajunta dos noms diferents
CORTE = set(",;:.()«»…!?")   # si el token acaba aquí, el nom acaba aquí

formas = defaultdict(Counter)
grupos_gt = Counter()        # cops que cada seqüència capitalitzada surt al GT
for frase in frases_gt:
    crudos = frase.split()
    tokens = [t.strip(',.;:¿?¡!()"\'«»…—-') for t in crudos]
    i = 1
    while i < len(tokens):
        if not tokens[i][:1].isupper():
            i += 1
            continue
        j = i
        while j < len(tokens) and tokens[j][:1].isupper() and (j - i) < MAX_TOKENS:
            j += 1
            if set(crudos[j - 1][-1:]) & CORTE:
                break
        if j - i >= 2:
            grupo = " ".join(tokens[i:j])
            grupos_gt[grupo] += 1
            for tok in tokens[i:j]:
                if norm(tok):
                    formas[norm(tok)][grupo] += 1
        i = max(j, i + 1)


def forma_completa(fragmento):
    f = formas.get(fragmento)
    return f.most_common(1)[0][0] if f else fragmento


# Diversos fragments col·lapsen en la mateixa entitat ('lamine' + 'yamal'): els seus
# senyals s'han d'acumular, no competir per separat dins de l'embut.
grupos = defaultdict(list)
for e in errores:
    grupos[forma_completa(e["bien_escrita"])].append(e)

print(f"{len(errores)} fragments amb error -> {len(grupos)} entitats reconstruïdes")
for ent, miembros in sorted(grupos.items(), key=lambda kv: -sum(
        n for m in kv[1] for _, n in m["mal_transcrita"]))[:10]:
    print(f"  {ent:32} <- {', '.join(m['bien_escrita'] for m in miembros)}")


338 fragments amb error -> 302 entitats reconstruïdes
  Lamine Yamal                     <- lamine, yamal
  Pérez Llorca                     <- llorca, pérez
  Núñez Feijóo                     <- feijóo, núñez
  Torre Blanca                     <- torre, blanca
  Les Corts                        <- corts
  Carlos Mazón                     <- mazón, carlos
  Dana Alice                       <- dana, alice
  Leire Diez                       <- diez
  Radio Nacional Radiogaceta       <- radiogaceta
  catarroja                        <- catarroja


In [5]:
# A.3 — Embut de descartes automàtics, aplicat a l'ENTITAT reconstruïda per A.2.
from num2words import num2words

SIM_MIN, LONG_MIN = 0.65, 4   # semblança mínima grafia+fonètica; longitud mínima

def es_cifra_de(correcta, v):
    """diez -> 10 és normalització, no error lèxic. Comprovar que la xifra ES la paraula
    evita descartar un dígit que ha coincidit per soroll d'alineament (club -> 1)."""
    return v.isdigit() and strip_accents(correcta) == strip_accents(num2words(int(v), lang="es"))

def n_tildes(p):
    return sum(a != b for a, b in zip(p, strip_accents(p)))

def es_tilde_del_modelo(correcta, v):
    """El model accentua i el GT no ('paris' -> 'parís'): qui està mal escrit és el GT."""
    return (v != OMITIDA and strip_accents(v) == strip_accents(correcta)
            and n_tildes(v) > n_tildes(correcta))

normalizadas = []   # variants tretes per no ser errors: es guarden per auditar-ho

def variantes_utiles(correcta, variantes):
    """Neteja per VARIANT, no per registre.

    Amb un `any()` sobre el registre, una sola variant normalitzada en tapava la resta:
    'alaves' -> ['alavés', 'a la vez'] queia sencera i es perdia "a la vez", que sí és
    una fallada real; i 'diez' -> ['10', 'díez'] s'enduia l'evidència de 'Leire Diez'.
    """
    utiles = []
    for v, n in variantes:
        if es_cifra_de(correcta, v):
            normalizadas.append({"entidad": correcta, "variante": v, "motivo": "cifra"})
        elif es_tilde_del_modelo(correcta, v):
            normalizadas.append({"entidad": correcta, "variante": v, "motivo": "gt_sin_tilde"})
        else:
            utiles.append((v, n))
    return utiles

def evidencia(miembros):
    """(fragments amb substitució creïble, si l'entitat s'ha arribat a ometre).

    La similitud es mesura sobre el FRAGMENT, que és el que `evaluate.py` va alinear,
    però la decisió es pren sobre l'entitat: n'hi ha prou que UN fragment porti
    evidència. 'earth' contra 'exalert' puntua 0.59 i queia sol, quan l'entitat és
    'Earth Alert'. Mesurat sobre les conservades, `sim_max` no separa el que el LLM
    després reconeix del que no (~0.92 de mitjana als dos costats), així que aquí no pot
    fer de filtre fi: només descarta el fragment que no aporta cap grafia comparable.
    """
    utiles, omitida = [], False
    for e in miembros:
        correcta = e["bien_escrita"]
        vs = variantes_utiles(correcta, e["mal_transcrita"])
        omitida = omitida or any(v == OMITIDA for v, _ in vs)
        subs = [v for v, _ in vs if v != OMITIDA]
        if not subs:
            continue
        sim = max(similitud(correcta, v) for v in subs)
        if sim >= SIM_MIN:
            utiles.append({"entidad": correcta, "veces": sum(n for _, n in vs),
                           "variantes": vs, "sim_max": round(sim, 3)})
    return utiles, omitida

def motivo_descarte(ent):
    """Nom comú i longitud, jutjats sobre la forma completa i no sobre el fragment."""
    tokens = [n for t in ent.split() if (n := norm(t))]
    # Una seqüència capitalitzada que es repeteix al GT és un nom propi encara que totes
    # les seves paraules siguin comunes: 'Televisión Española', 'Pacto Verde'.
    if grupos_gt[ent] < 2 and tokens and all(
            uso_minus[t] >= max(1, uso_mayus[t] * 0.5) for t in tokens):
        return "nombre_comun"          # surt en minúscula al GT
    if len(ent.replace(" ", "")) < LONG_MIN:
        return "muy_corta"
    return None

fuente_a, descartes, solo_omitida = [], defaultdict(list), []
for ent, miembros in grupos.items():
    utiles, omitida = evidencia(miembros)
    resumen = {"entidad": ent,
               "veces": sum(n for m in miembros for _, n in m["mal_transcrita"]),
               "fragmentos": [m["bien_escrita"] for m in miembros],
               "variantes": sorted({v for m in miembros for v, _ in m["mal_transcrita"]})}
    if motivo := motivo_descarte(ent):
        descartes[motivo].append(resumen)
    elif utiles:
        for d in utiles:
            d["forma_completa"] = ent
            fuente_a.append(d)
    elif omitida:
        # Omissió pura: Whisper s'ha saltat l'entitat sencera. És la fallada MÉS greu,
        # però no deixa cap grafia amb què validar-la, i Font A entra al dataset sense
        # round-trip. En comptes de tirar-les, es deixen llestes per mesurar-les.
        solo_omitida.append(resumen)
    else:
        descartes["sin_evidencia"].append(resumen)

for motivo, items in sorted(descartes.items(), key=lambda kv: -len(kv[1])):
    print(f"{motivo:22} {len(items):4}   {', '.join(d['entidad'] for d in items[:5])}")
print(f"{'a round-trip':22} {len(solo_omitida):4}   "
      f"{', '.join(d['entidad'] for d in solo_omitida[:5])}")
print(f"{'(variants normalitz.)':22} {len(normalizadas):4}   "
      + ", ".join(f"{d['entidad']}->{d['variante']}" for d in normalizadas[:5]))
print(f"{'CONSERVADES':22} {len(fuente_a):4} fragments -> "
      f"{len({d['forma_completa'] for d in fuente_a})} entitats")

for d in fuente_a:
    reg = candidatos[d["forma_completa"]]
    reg["fuentes"].add("A_errores_modelo")
    reg["freq"] += d["veces"]
    reg["veces_error"] = reg.get("veces_error", 0) + d["veces"]
    reg["variantes_erroneas"] = sorted(set(reg.get("variantes_erroneas", []))
                                       | {v for v, _ in d["variantes"] if v != OMITIDA})

# Els descartes es guarden per poder auditar l'embut sense re-executar evaluate.py.
(ENT / "entidades_fuente_a.json").write_text(llm.json_compacte({
    "conservadas": sorted(fuente_a, key=lambda d: -d["veces"]),
    "solo_omitida": solo_omitida,
    "variantes_normalizadas": normalizadas,
    "descartadas": {m: v for m, v in descartes.items()}}), encoding="utf-8")

# Verificable amb `src/verify_entities.py --input {RUTA_A_OMITIDES}` (veure la
# cel·la de fusio final: hi entra amb provinenca 'A_omitida_verificada', no com si
# fos Font B). Es la fallada mes greu -- omissio total -- i no te grafia amb que
# validar-se per LLM, aixi que el round-trip fa aquesta funcio.
RUTA_A_OMITIDES.write_text(
    llm.json_compacte([{"entidad": d["entidad"], "veces": d["veces"]} for d in solo_omitida]),
    encoding="utf-8")

print(f"\n{len(candidatos)} entitats úniques cap a la validació")
for ent, d in sorted(candidatos.items(), key=lambda kv: -kv[1]["veces_error"])[:15]:
    print(f"  {ent:28} x{d['veces_error']:<3} [{', '.join(d['variantes_erroneas'][:4])}]")


sin_evidencia            25   Leire Diez, Paris Fútbol Club, Iñigo Pérez, genova, Luis Valles
nombre_comun             22   Veinticuatro Horas, La Vida, Secretario General, día, presidente
muy_corta                 2   tes, man
a round-trip             16   Miriam García Navarro, Radio España, Isabel Díaz Ayuso, consell, The Floor
(variants normalitz.)    17   diez->10, diez->díez, veinticuatro->24, paris->parís, iñigo->íñigo
CONSERVADES             259 fragments -> 237 entitats

237 entitats úniques cap a la validació
  Lamine Yamal                 x41  [alhamid, alhamid jamal, cono sin la miña mal, jamal]
  Pérez Llorca                 x24  [de yorca, peret, pérez llota mazón de verdad, yerka]
  Núñez Feijóo                 x22  [contrafijo, feijo, feijo dijo, feijó]
  Torre Blanca                 x22  [a, ahí por ventorra, blancos, todos los]
  Les Corts                    x15  [al scorch, andres kors, cors, corvalencianes]
  Carlos Mazón                 x13  [amazon, amazon emilian

### Validació de grafia amb LLM

L'embut automàtic no pot decidir si la grafia correcta és `vasconia` o `Baskonia`: això
és coneixement del món. Dues coses el fan funcionar: se li adjunta **la frase del GT** on
apareix l'entitat, i se li permet **dir que no ho sap** en comptes d'inventar una grafia
per a un poble petit o un periodista local.


In [6]:
class IndexFrases:
    """Frases d'un corpus amb la seva versió plana. Les dues llistes han d'anar juntes.
    Permite buscar palabras a gran velocidad ignorando tildes y mayúsculas, 
    pero le devuelve al LLM el fragmento de texto con su ortografía y puntuación intactas."""

    def __init__(self, frases):
        self.frases = frases
        self.planas = [strip_accents(f.lower()) for f in frases]

    def contexto(self, entidad, ventana=220):
        claves = [strip_accents(norm(entidad))] + sorted(
            (strip_accents(t) for t in norm(entidad).split()), key=len, reverse=True)
        for clave in claves:
            patron = re.compile(rf"\b{re.escape(clave)}\b")
            for frase, plana in zip(self.frases, self.planas):
                if m := patron.search(plana):
                    ini = max(0, m.start() - ventana // 2)
                    return frase[ini:ini + ventana].strip()
        return ""

def validar_grafias(items, system_prompt, mida_lot=25, model=MODEL_VALIDACION):
    """Cada entrada viatja amb un `id` i la resposta ha de portar els mateixos ids en el
    mateix ordre. """
    client, _ = llm.client_per_model(model)
    resultados, coste = [], 0.0
    for i, lote in enumerate(llm.per_lots(items, mida_lot), start=1):
        payload = json.dumps([{"id": j, **it} for j, it in enumerate(lote)], ensure_ascii=False)
        obj, meta = llm.crida_amb_reintents(
            client, model, system_prompt, f"Entidades a validar:\n{payload}",
            llm.SCHEMA_VALIDACIO, "validacion_entidades", max_tokens=8192)
        llm.validar_ids(obj["entidades"], len(lote))
        resultados += [dict(r, entrada=lote[r["id"]]["entrada"]) for r in obj["entidades"]]
        coste += llm.cost(meta, model)
        print(f"  lot {i}: {len(lote)} entitats" + (f"  (${coste:.4f})" if coste else ""))
    return resultados

def tipo_cambio(entrada, correcta):
    """El flag `gt_erroneo` del LLM no basta: les entitats venen en minúscula normalitzada
    i una simple capitalització el dispara.

    La comparacio d'expansio va sobre `phonetics.clau()`, no sobre `strip_accents`: aquell
    conserva la puntuacio, i per un guionet `García Page` -> `Emiliano García-Page` queia
    a 'grafia_distinta' i el guardarrail de similitud la tombava (0.547) tot i ser una
    expansio correcta.
    """
    a, b = entrada.strip().lower(), correcta.strip().lower()
    if a == b:
        return "solo_mayusculas"
    if strip_accents(a) == strip_accents(b):
        return "acento"
    ka, kb = phonetics.clau(entrada), phonetics.clau(correcta)
    if ka == kb:
        return "acento"
    if f" {ka} " in f" {kb} " or f" {kb} " in f" {ka} ":
        return "expansion"
    return "grafia_distinta"

In [7]:
# A.4 — Validar Font A  (CONSUMEIX el model)
index_gt = IndexFrases(frases_gt)
a_validar = [{"entrada": ent, "variantes_whisper": d["variantes_erroneas"][:8],
              "veces": d["veces_error"], "contexto": index_gt.contexto(ent)}
             for ent, d in candidatos.items()]
print(f"{len(a_validar)} entitats a validar amb {MODEL_VALIDACION}")
validaciones = validar_grafias(a_validar, llm.SYSTEM_VALIDACIO_A)


237 entitats a validar amb gpt-4.1-mini
  lot 1: 25 entitats  ($0.0030)
  lot 2: 25 entitats  ($0.0062)
  lot 3: 25 entitats  ($0.0092)
  lot 4: 25 entitats  ($0.0123)
  lot 5: 25 entitats  ($0.0154)
  lot 6: 25 entitats  ($0.0182)
  lot 7: 25 entitats  ($0.0213)
  lot 8: 25 entitats  ($0.0244)
  lot 9: 25 entitats  ($0.0272)
  lot 10: 12 entitats  ($0.0288)


In [9]:
# A.5 — Aplicar la validació.
#
# NO es filtra per `tipo_cambio`: com que `evaluate.py` normalitza a minúscules,
# QUALSEVOL entitat que el LLM només hagi de capitalitzar cau a 'solo_mayusculas', i
# descartar-les es carregava 136 de 178 entitats (entre elles 'Lamine Yamal').
#
# El guardarraïl que sí serveix és `diagnosticar_cambio`: `sim_sustitucion` mira
# només els trossos que canvien, no la similitud global, que les paraules compartides
# inflaven. Detecta quan
# el LLM "corregeix" substituint l'entitat per una altra ('Japoel Tel Aviv' -> 'Maccabi
# Tel Aviv', dos clubs distints: 0.70 global, 0.44 mirant el que canvia).
SIM_CAMBIO_MIN = 0.65

for v in validaciones:
    v["tipo_cambio"] = tipo_cambio(v["entrada"], v["grafia_correcta"])
    v["diagnostico_cambio"] = diagnosticar_cambio(v["entrada"], v["grafia_correcta"])
    v["sim_cambio"] = v["diagnostico_cambio"]["sim_sustitucion"]

# Una expansio ('García Page' -> 'Emiliano García-Page') afegeix un nom que abans no
# hi era, i `sim_sustitucion` mira nomes els trossos que se SUBSTITUEIXEN: en una
# expansio comparava el cognom sol contra el nom sencer i el tombava (0.547). El que
# cal comprovar en una expansio no es la similitud sino que no s'hagi perdut res del
# ground truth, i aixo ho diu `eliminados`.
def sospitosa(v):
    d = v["diagnostico_cambio"]
    if v["tipo_cambio"] == "expansion":
        return bool(d["eliminados"])
    return v["sim_cambio"] < SIM_CAMBIO_MIN

es_entidad = [v for v in validaciones if v["tipo"] != "NO_ENTIDAD"]
validadas = [v for v in es_entidad if v["entidad_conocida"] and not sospitosa(v)]
revisar = [v for v in es_entidad if v not in validadas]
rechazadas = [v for v in validaciones if v["tipo"] == "NO_ENTIDAD"]

print(f"validades {len(validadas)} | revisió humana {len(revisar)} | NO_ENTIDAD {len(rechazadas)}")
print("\n--- Correccions reals (el GT estava mal) ---")
for v in validadas:
    if v["tipo_cambio"] != "solo_mayusculas":
        print(f"  [{v['tipo_cambio']:15}] {v['entrada']:24} -> {v['grafia_correcta']}")
print(f"\n--- Sospitoses: el LLM ha canviat l'entitat, no la grafia ---")
for v in sorted((x for x in es_entidad if sospitosa(x)), key=lambda x: x["sim_cambio"]):
    perdut = v["diagnostico_cambio"]["eliminados"]
    motiu = f"perd {perdut}" if perdut else f"sim {v['sim_cambio']:.2f}"
    print(f"  [{motiu:16}] {v['entrada']:22} -> {v['grafia_correcta']}")

(ENT / "entidades_fuente_a_validadas.json").write_text(llm.json_compacte(
    {"validadas": validadas, "revisar_humano": revisar, "rechazadas": rechazadas}),
    encoding="utf-8")


validades 136 | revisió humana 95 | NO_ENTIDAD 6

--- Correccions reals (el GT estava mal) ---
  [expansion      ] Núñez Feijóo             -> Alberto Núñez Feijóo
  [expansion      ] Dana Alice               -> Dana
  [expansion      ] Díaz Ayuso               -> Isabel Díaz Ayuso
  [grafia_distinta] Japoel Tel Aviv          -> Hapoel Tel Aviv
  [grafia_distinta] vasconia                 -> Baskonia
  [grafia_distinta] Lorenzo Mouseti          -> Lorenzo Musetti
  [acento         ] baldovi                  -> Baldoví
  [grafia_distinta] letour                   -> Letur
  [acento         ] valenciá                 -> Valencia
  [grafia_distinta] Laszlo Krasnáorkai       -> László Krasznahorkai
  [grafia_distinta] Laszlo Krasnáhorkai      -> László Krasznahorkai
  [expansion      ] Confederación Hidrográfica -> Confederación Hidrográfica del Júcar
  [grafia_distinta] Festival Eña             -> Festival Eñe
  [grafia_distinta] Alberto Núñez Fejo       -> Alberto Núñez Feijóo
  [expansi

100125

## Font B — NER automàtic sobre corpus (RTVE / Wikipedia)

El NER troba entitats, no errors, i la majoria (`Gobierno`, `Estado`) Whisper les
transcriu perfectament. `motivo_descarte_b` treu el soroll que hi deixa: spans que
travessen un salt de frase (`Ucrania\nEfectivamente`), sintagmes comuns sense cap
paraula capitalitzada i verbs o pronoms colats al span (`Athletic Club superó`).


In [10]:
import spacy

# 'md' i no 'sm': sobre aquest corpus arregla el que el filtre automàtic no pot enxampar
# -- spans que es mengen una paraula de més ('Cucurella y Baena') i falsos positius
# gramaticals (Existe/Després) que 'sm' marcava per anar a principi de frase.
nlp = spacy.load("es_core_news_md")
LABELS, MAX_TOKENS_ENT = {"PER", "ORG", "LOC"}, 4
POS_NO_ENTIDAD = {"VERB", "AUX", "PRON", "ADV", "SCONJ", "CCONJ", "INTJ"}
candidatos_b = defaultdict(lambda: {"freq": 0, "docs": set(), "labels": Counter()})


def clave_entidad(ent):
    """spaCy a vegades inclou el punt final a l'span i 'Gobierno.' entrava com un
    candidat distint de 'Gobierno'. El punt SÍ es conserva a les abreviatures
    ('EE.UU.'), que es reconeixen perquè ja porten un altre punt a dins."""
    texto = ent.text.strip()
    sin_final = texto.rstrip(',.;:¿?¡!()"\'«»…—- ')
    if texto.endswith(".") and "." in sin_final:
        return sin_final + "."
    return sin_final.lstrip(',.;:¿?¡!()"\'«»…—- ')


# Tots els fitxers, amb extensió o sense: un `rglob("*.txt")` es saltava en silenci els
# documents desats sense extensió i 1 de cada 4 no arribava al NER.
dir_corpus = ARREL / "lab/proves_inicials/proves/raw_text"
entidades_ner, textos_corpus = [], []
for p in sorted(x for x in dir_corpus.rglob("*") if x.is_file() and not x.name.startswith(".")):
    texto = p.read_text(encoding="utf-8", errors="ignore")
    textos_corpus.append(texto)
    # Es guarda l'Span, no el text: el filtre necessita el POS i els límits de frase.
    entidades_ner += [(p.name, e) for e in nlp(texto).ents if e.label_ in LABELS]
print(f"corpus: {len(textos_corpus)} fitxers, {len(entidades_ner)} spans")

may_b, min_b = Counter(), Counter()
for frase in frases_de(textos_corpus):
    for i, palabra in enumerate(frase.split()):
        tok = palabra.strip(',.;:¿?¡!()"\'«»…—-')
        if tok and i > 0 and (n := norm(tok)):
            (may_b if tok[0].isupper() else min_b)[n] += 1


def capitalizada_o_conector(t):
    """L'elisió catalana (`d'Empúries`) no compta com a paraula sense capitalitzar: el
    que importa és com comença el que segueix l'apòstrof."""
    if t[:1].isupper() or norm(t) in STOP:
        return True
    m = re.match(r"^[dlsn]'(.+)$", t, re.IGNORECASE)
    return bool(m and m.group(1)[:1].isupper())


def motivo_descarte_b(ent):
    texto = clave_entidad(ent)
    palabras, toks = texto.split(), norm(texto).split()
    if not toks or len(toks) > MAX_TOKENS_ENT:
        return "longitud"
    # 'De la Fuente' comença per stopword però ÉS l'entitat: només es descarta si, tret
    # el connector inicial, no queda cap paraula capitalitzada.
    if toks[0] in STOP and not any(p[:1].isupper() for p in palabras[1:]):
        return "empieza_stop"
    if len(toks) == 1:
        # Sigles reals ('UE', 'G7') van senceres en majúscula, a diferència d'un
        # fragment capitalitzat per posició ('Així').
        if len(toks[0]) < 4 and not palabras[0].isupper():
            return "muy_corta"
        if min_b[toks[0]] >= max(1, may_b[toks[0]] * 0.5):
            return "nombre_comun"
    if "\n" in ent.text or any(t.is_sent_start for t in ent[1:]):
        return "cruza_frase"
    # Excepció: topònims amb nucli genèric en minúscula ('estrecho de Ormuz').
    a_revisar = palabras
    if len(palabras) >= 3 and not palabras[0][:1].isupper() and palabras[1].lower() in ("de", "del"):
        a_revisar = palabras[1:]
    if not all(capitalizada_o_conector(t) for t in a_revisar):
        return "no_capitalizado"
    return "pos_no_nominal" if any(t.pos_ in POS_NO_ENTIDAD for t in ent) else None


descartes_b = defaultdict(list)
for doc, ent in entidades_ner:
    if motivo := motivo_descarte_b(ent):
        descartes_b[motivo].append(clave_entidad(ent) or ent.text.strip())
    elif texto := clave_entidad(ent):
        candidatos_b[texto]["freq"] += 1
        candidatos_b[texto]["docs"].add(doc)
        # L'etiqueta es vota entre totes les aparicions: assignar-la solta feia guanyar
        # sempre l'ÚLTIMA ocurrència, encara que fos la minoritària.
        candidatos_b[texto]["labels"][ent.label_] += 1

for motivo, items in sorted(descartes_b.items(), key=lambda kv: -len(kv[1])):
    print(f"{motivo:16} {len(items):4}   {', '.join(items[:5])}")
for d in candidatos_b.values():
    d["label"] = d["labels"].most_common(1)[0][0]
print(f"\ncandidats B únics: {len(candidatos_b)}")


corpus: 4 fitxers, 355 spans
nombre_comun        8   Récords, Roja, Estamos, Estado, Además
cruza_frase         5   Ucrania
Efectivamente, Gobierno, Gobierno, Gobierno, Organización
Los miembros del
pos_no_nominal      3   O Rosal, Pasamos, Sumar
no_capitalizado     3   presidente Trump, Gobierno central, Constitución de 1978
longitud            2   Centre Delàs de Estudios para la Paz, Comisión General de Secretarios de Estado y Subsecretarios

candidats B únics: 167


In [13]:
# Formes parcials: 'Ronaldo' i 'Cristiano Ronaldo' són el mateix referent i gasten dues
# validacions i dues verificacions. No es fusionen (Whisper pot fallar només en una),
# però s'anoten. Només PERSONES: per a LLOC/ORG compartir una paraula no implica el
# mateix referent ('Almería' vs 'Almería Aeropuerto').
for corta, d in candidatos_b.items():
    if len(corta.split()) == 1 and d["label"] == "PER":
        largas = [l for l in candidatos_b if l != corta and corta in l.split()
                  and candidatos_b[l]["label"] == "PER"]
        if largas:
            d["forma_larga"] = max(largas, key=lambda l: candidatos_b[l]["freq"])

# Font C — el BPE de Whisper trenca les paraules poc freqüents en subtokens: com més
# subtokens per caràcter, menys probable que el model hagi vist la paraula sencera. Es
# fa servir el tokenizer REAL: Whisper va refer el BPE per a multilingüe i el recompte
# difereix respecte de gpt2 justament en els noms estrangers, que són l'objectiu.
# Ordena, no selecciona: en normalitzar per caràcters premia les paraules curtes i puja
# les sigles per sobre de 'Krasznahorkai'.
from transformers import WhisperTokenizer

tok = WhisperTokenizer.from_pretrained(
    "openai/whisper-large-v3", cache_dir=ARREL / ".hf_cache/hub")
for ent, d in candidatos_b.items():
    d["score_fragmentacion"] = round(len(tok.encode(ent, add_special_tokens=False)) / len(ent), 3)

ranking_b = sorted(candidatos_b.items(), key=lambda kv: -kv[1]["score_fragmentacion"])
(ENT / "entidades_fuente_b_ranked.json").write_text(llm.json_compacte({
    ent: {"freq": d["freq"], "label": d["label"], "docs": sorted(d["docs"]),
          "score_fragmentacion": d["score_fragmentacion"], "forma_larga": d.get("forma_larga")}
    for ent, d in ranking_b}), encoding="utf-8")
for ent, d in ranking_b[:15]:
    print(f"  {d['score_fragmentacion']:.3f}  {ent:34} [{d['label']}] x{d['freq']}")


  1.000  G7                                 [ORG] x1
  0.800  CIDOB                              [ORG] x3
  0.750  Pelé                               [PER] x1
  0.750  Irán                               [LOC] x1
  0.750  RTVE                               [ORG] x1
  0.750  PSOE                               [ORG] x1
  0.667  Mbappé                             [PER] x1
  0.667  EE.UU.                             [LOC] x6
  0.600  Araba                              [LOC] x1
  0.600  Álava                              [LOC] x1
  0.600  AEMET                              [ORG] x1
  0.600  Cádiz                              [LOC] x3
  0.600  Pedri                              [PER] x2
  0.600  Rusia                              [LOC] x1
  0.600  Rodri                              [PER] x1


In [14]:
# B.4 — Validar Font B  (CONSUMEIX el model)
# Ordre de `ranking_b` (Font C: pitjor fragmentacio primer), no de `candidatos_b`: es
# el que fa que ordenar realment SERVEIXI d'alguna cosa. Abans `b_validar` iterava
# `candidatos_b.items()` (ordre d'aparicio al NER) i `entidades_fuente_b_ranked.json`
# nomes es guardava, no es tornava a llegir enlloc -- si mai es fa servir `--limit` a
# `verify_entities.py`, ara gasta el pressupost en els candidats mes dificils primer.
index_b = IndexFrases(frases_de(textos_corpus))
b_validar = [{"entrada": ent, "label_ner": d["label"], "veces": d["freq"],
              "contexto": index_b.contexto(ent)} for ent, d in ranking_b]
print(f"{len(b_validar)} entitats de Font B a validar")
validaciones_b = validar_grafias(b_validar, llm.SYSTEM_VALIDACIO_B)

for v in validaciones_b:
    v["forma_larga"] = candidatos_b.get(v["entrada"], {}).get("forma_larga")
validadas_b = [v for v in validaciones_b if v["tipo"] != "NO_ENTIDAD" and v["entidad_conocida"]]
revisar_b = [v for v in validaciones_b if v["tipo"] != "NO_ENTIDAD" and not v["entidad_conocida"]]
rechazadas_b = [v for v in validaciones_b if v["tipo"] == "NO_ENTIDAD"]

(ENT / "entidades_fuente_b_validadas.json").write_text(llm.json_compacte(
    {"validadas": validadas_b, "revisar_humano": revisar_b, "rechazadas": rechazadas_b}),
    encoding="utf-8")
print(f"a verificar {len(validadas_b)} | revisió humana {len(revisar_b)} | "
      f"NO_ENTIDAD {len(rechazadas_b)}")
print("\nSegüent: dictionary.ipynb (ETAPA='treball') i després src/verify_entities.py")


167 entitats de Font B a validar
  lot 1: 25 entitats  ($0.0032)
  lot 2: 25 entitats  ($0.0061)
  lot 3: 25 entitats  ($0.0091)
  lot 4: 25 entitats  ($0.0121)
  lot 5: 25 entitats  ($0.0151)
  lot 6: 25 entitats  ($0.0179)
  lot 7: 17 entitats  ($0.0200)
a verificar 157 | revisió humana 8 | NO_ENTIDAD 2

Següent: dictionary.ipynb (ETAPA='treball') i després src/verify_entities.py


## Fusió final

**Aquesta cel·la és independent de la resta**: llegeix dels fitxers, no de la memòria del
kernel, així que després del round-trip pots re-executar-la sola sense repetir cap
validació de LLM.

| Font | Criteri d'entrada | Prioritat |
|---|---|---|
| **A** | fallada observada sobre àudio **i** el LLM reconeix l'entitat | 10 + fragmentació |
| **B** | `tasa_error >= UMBRAL` **mesurada** pel round-trip | 1 + `tasa_error` |

Sense round-trip encara, només exporta Font A i avisa. `score_fragmentacion` **no** s'usa
com a llindar: provat a `>= 0.6`, de 12 seleccionades només 4 fallaven de veritat i
deixava fora les nou amb `tasa_error = 1.0`.


In [3]:
UMBRAL_TASA_ERROR = 0.3

# Entitats de Font A que el LLM no reconeix però que sí són reals. Sense aquesta repesca
# es perdrien; sense el filtre `entidad_conocida` hi entra tot el soroll del ground truth
# ('abracabalcán', 'huíkop', 'mediarres'), que no són entitats de cap mena.
REPESCA = {
    "Japoel Tel Aviv": "Hapoel Tel Aviv",   # no confondre amb Maccabi: clubs diferents
    "Laszlo Krasnáhorkai": "László Krasznahorkai",
    "lladro": "Lladró", "Pau Cubarsí": "Pau Cubarsí", "Salomé Pradas": "Salomé Pradas",
    "Juanfran Pérez Llorca": "Juanfran Pérez Llorca", "Pepa Millán Vox": "Pepa Millán",
    "Esther Muñoz PP": "Esther Muñoz", "Morero Bonilla": "Moreno Bonilla",
    "José Català": "José Català", "Isabel Gémio": "Isabel Gémio",
    "Pete Hegseth": "Pete Hegseth", "mazones": "Mazón", "tofás": "Tofaş", "íñigo": "Iñigo",
}

# Font A des del fitxer: diversos fragments col·lapsen en la mateixa entitat canònica.
font_a = defaultdict(lambda: {"freq": 0, "veces_error": 0, "variantes_erroneas": set()})
for d in json.loads((ENT / "entidades_fuente_a.json").read_text("utf-8"))["conservadas"]:
    reg = font_a[d.get("forma_completa") or d["entidad"]]
    reg["freq"] += d["veces"]
    reg["veces_error"] += d["veces"]
    reg["variantes_erroneas"] |= {v for v, _ in d["variantes"] if v != OMITIDA}

val_a = json.loads((ENT / "entidades_fuente_a_validadas.json").read_text("utf-8"))
canon = {v["entrada"]: v["grafia_correcta"] for v in val_a["validadas"]} | REPESCA

# `claus_salida` indexa per `phonetics.clau()`, no per string exacte: Font A i Font B
# criden LLMs independents (SYSTEM_VALIDACIO_A / SYSTEM_VALIDACIO_B) i no hi ha cap
# garantia que tornin la MATEIXA grafia (accent, caixa) per al mateix referent real.
# Sense normalitzar, una entitat que aparegui a totes dues fonts acabava amb dues
# entrades a `entidades_candidatas.json` -- doble crida LLM i, pitjor, una col·lisió
# silenciosa a l'índex de `phonetics.Diccionari` (nomes avisa per pantalla).
salida, excluidas = {}, []
claus_salida: dict[str, str] = {}   # clau normalitzada -> grafia real usada a `salida`
for ent, d in font_a.items():
    if ent not in canon:
        excluidas.append(ent)
        continue
    entrada = {"fuentes": ["A_errores_modelo"], "freq": d["freq"], "prioridad": 10.0,
               "veces_error": d["veces_error"],
               "variantes_erroneas": sorted(d["variantes_erroneas"])}
    c = canon[ent]
    if c != ent:
        entrada["grafia_original"] = ent
    k = phonetics.clau(c)
    if k in claus_salida:
        anterior = salida[claus_salida[k]]
        anterior["freq"] += entrada["freq"]
        anterior["veces_error"] += entrada["veces_error"]
        anterior["variantes_erroneas"] = sorted(set(anterior["variantes_erroneas"])
                                                | set(entrada["variantes_erroneas"]))
        anterior.setdefault("grafies_originals", []).append(ent)
    else:
        claus_salida[k] = c
        salida[c] = entrada
print(f"Font A: {len(salida)} reconegudes | {len(excluidas)} excloses "
      f"(el LLM no les reconeix; repesca-les a REPESCA)")
for e in sorted(excluidas):
    print(f"    {e}")

# Font B des del fitxer, no del kernel: aquesta cel·la esta feta per executar-se sola
# despres del round-trip, i llavors `candidatos_b` no existeix -- `freq` sortia 0 per a
# TOTES les entitats de Font B.
ruta_ranked = ENT / "entidades_fuente_b_ranked.json"
ranked_b = json.loads(ruta_ranked.read_text("utf-8")) if ruta_ranked.exists() else {}

# Font B: només les que el round-trip ha MESURAT que fallen.
ruta_rt = ENT / "entidades_verificades_roundtrip.json"
if ruta_rt.exists():
    rt = json.loads(ruta_rt.read_text("utf-8"))
    # Les ÀNCORES són instrumentació de mesura: una àncora que falla vol dir que la
    # condició està mal calibrada, no que sigui una entitat a reforçar.
    medidas = {r["entitat"]: r for r in rt["resultats"]
               if r.get("tasa_error") is not None and not r.get("ancora")}
    if cal := rt["_meta"].get("ancores"):
        print(f"\nCalibració: les {cal['n']} àncores fallen {cal['tasa_error_mitjana']:.2f} "
              f"de mitjana. El llindar {UMBRAL_TASA_ERROR} es llegeix sobre aquesta base.")
    n, n_a_omitida = 0, 0
    for ent, r in medidas.items():
        k = phonetics.clau(ent)
        if r["tasa_error"] < UMBRAL_TASA_ERROR or k in claus_salida:
            continue
        # Provinenca real, no "tot el que no es Font A es Font B": si aquesta entitat
        # ve de `RUTA_A_OMITIDES` (omissio pura, la fallada mes greu de Font A, mesurada
        # per round-trip en lloc de per substitucio), ha d'entrar amb la seva prioritat
        # de Font A -- sense aixo quedava etiquetada com 'B_ner_corpus_verificada' nomes
        # perque compartia fitxer de sortida amb la Font B de veritat.
        es_a_omitida = r.get("font_input") == RUTA_A_OMITIDES.name
        b = ranked_b.get(ent, {})
        salida[ent] = {
            "fuentes": ["A_omitida_verificada"] if es_a_omitida else ["B_ner_corpus_verificada"],
            "freq": b.get("freq", 0),
            "tasa_error": r["tasa_error"],
            # Desempata els molts candidats que comparteixen tasa_error.
            "distancia_mitjana": r.get("distancia_mitjana"),
            "prioridad": round((10 if es_a_omitida else 1) + r["tasa_error"]
                               + 0.1 * (r.get("distancia_mitjana") or 0), 3),
        }
        claus_salida[k] = ent   # perque duplicats dins de la propia Font B tambe es detectin
        n += 1
        n_a_omitida += es_a_omitida
    print(f"Font B: {n - n_a_omitida} injectades | Font A omesa: {n_a_omitida} injectades "
          f"(tasa_error >= {UMBRAL_TASA_ERROR}, de {len(medidas)} verificades)")
else:
    print(f"\nSense round-trip: només Font A. Genera'l amb python3 src/verify_entities.py")

salida = dict(sorted(salida.items(), key=lambda kv: -kv[1]["prioridad"]))
(ENT / "entidades_candidatas.json").write_text(llm.json_compacte(salida), encoding="utf-8")
print(f"\n{len(salida)} entitats candidates -> entidades_candidatas.json")


Font A: 128 reconegudes | 98 excloses (el LLM no les reconeix; repesca-les a REPESCA)
  Aitor Caranca, Amparo Climent, Ana Valmoral, Andrés Sehobia, Antonio Molas, Armón Roy, Bo Berrisa, Chris Bosch, Costas Cádiz, Dean Hauser, Earth Alert, Encarna Assa, Encarnas Sánchez, Ernest Schilders, García Page...

Calibració: les 5 àncores fallen 0.00 de mitjana. El llindar 0.3 es llegeix sobre aquesta base.
Font B: 33 injectades (tasa_error >= 0.3, de 144 verificades)

161 entitats candidates -> entidades_candidatas.json
